<a href="https://colab.research.google.com/github/ronronrivera/Build-a-Large-Language-Model-From-Scratch-/blob/main/chapter1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
import os
import urllib.request

In [78]:
if not os.path.exists('the-verdict.txt'):
  url = ('https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt')
  file_path = 'the-verdict.txt'
  urllib.request.urlretrieve(url, file_path)

In [79]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
  raw_text = f.read()



**BYTE PAIR ENCODING**

---



In [80]:
import tiktoken

In [81]:
tiktoken.__version__

'0.14.0'

In [82]:
tokenizer = tiktoken.get_encoding('gpt2')

In [83]:
encoder = tokenizer.encode("asdasdasd")
print(encoder)

[292, 67, 292, 67, 292, 67]


***DATA SAMPLING WITH A SLIDING WINDOW***

In [84]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [85]:
enc_sample = enc_text[50:]

In [86]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:       {y}")

x: [290, 4920, 2241, 287]
y:       [4920, 2241, 287, 257]


In [87]:
import torch

In [88]:
torch.__version__

'2.14.1+cpu'

In [89]:
print(torch.cuda.is_available())

False


In [90]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):

    self.input_ids = []
    self.target_ids = []

    #tokenize entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    #use sliding window to chunk the book into overlapping sequence of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chuck = token_ids[i + 1: max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chuck))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [91]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):

  #initialize the tokenizer
  tokenizer = tiktoken.get_encoding('gpt2')

  #create dataset
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

  return dataloader

In [92]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
  raw_text = f.read()



In [93]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [94]:
seconds_batch = next(data_iter)
print(seconds_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([], size=(1, 0))]


In [95]:
dataloader = create_dataloader_v1(raw_text,  batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print(f"Inputs:\n {inputs}")
print(f"Target:\n {targets}")

RuntimeError: stack expects each tensor to be equal size, but got [4] at entry 0 and [0] at entry 1